# 🦖 Example 6: Deep-Time Paleo-Globes, Full-Spectrum Filament Mixing & 3D Preview

This tutorial explains how to create physical, 3D printable globes of ancient Earth across deep geological time (0 to 540 Ma). It covers plate tectonic reconstructions, Scotese PALEOMAP PaleoDEM elevation grids, physics-based full-spectrum filament mixing for multi-material FDM 3D printing, discrete banded colormaps, and 2D/3D visualization and rendering.

---

### 🌎 Key Capabilities Demonstrated:
1. **Paleo Datasets & Catalog**: Querying 108 geological time slices from the Cambrian (540 Ma) to modern day via `globe3d.datasets.paleo`.
2. **Deep-Time Coordinate Tracking**: Tracing modern landmarks (e.g. Oxford, UK and Baltimore, USA) across continental drift using `PlateModel`.
3. **Full-Spectrum Filament Mixing**: Predicting perceived optical layer mixing colors from physical base spools (CMYKW) using a Kubelka-Munk polynomial model (as per *Full Spectrum 3D* / *OrcaSlicer-FullSpectrum* / *Prusa ColorMix*).
4. **Banded Hypsometric Colormaps**: Constructing discrete 8-zone elevation colormaps matched to exact filament mix recipes.
5. **Texture Preprocessing & Stamping**: Eliminating hillshade baked shadows, stamping spherical SVG markers at reconstructed coordinates, and adding North/South polar markers.
6. **3D Displaced Globe Generation**: Building hollowed, vertically exaggerated spherical models with coastline shelf steps via `GlobeModel.from_paleo`.
7. **2D & 3D Rendering and Preview**: Visualizing discrete textures in 2D and orthographic 3D mesh perspectives in Matplotlib, plus interactive Trimesh preview.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import globe3d
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    ImageColourer,
    PlateModel,
    datasets,
    mix_filament_pair,
    mix_filaments,
    create_filament_palette,
    match_filament_color,
    create_banded_colormap,
    create_elevation_banded_colormap,
    apply_banded_colormap,
    plot_filament_palette,
    plot_banded_colormap_preview,
    plot_globe_preview,
    undo_hillshading,
    add_svg_symbol_spherical,
    add_pole_markers,
)

print(f"globe3d version: {getattr(globe3d, '__version__', '0.1.0')}")

## 1. Inspecting the Scotese PALEOMAP Catalog
`globe3d` includes a registry of 108 reconstructed time slices based on Scotese & Wright (2018) PALEOMAP PaleoDEMs and matching PaleoAtlas textures.

In [ ]:
maps = datasets.paleo.list_scotese_maps()
print(f"Total available geological time slices: {len(maps)}")

print("\nSample time slices:")
for m in maps[::12]:  # Sample every 12th map
    print(f"  {m['age_ma']:>5.1f} Ma: Map {m['map_id']:02d} — {m['name']}")

## 2. Plate Tectonic Reconstructions: Tracing Modern Cities
Using the PALEOMAP Global Plate Model, we can trace where modern cities were situated millions of years ago.
Let's track **Oxford, UK** (51.75°N, -1.26°E) and **Baltimore, USA** (39.29°N, -76.61°W) at the **Permo-Triassic Boundary (250 Ma)**, when all major landmasses were merged into the supercontinent **Pangaea**:

In [ ]:
pm = datasets.paleo.scotese_plate_model()

oxford_modern = [51.7519, -1.2578]
baltimore_modern = [39.2904, -76.6122]

oxford_250ma = pm.reconstruct_points(oxford_modern, age_ma=250.0)[0]
baltimore_250ma = pm.reconstruct_points(baltimore_modern, age_ma=250.0)[0]

print("Oxford, UK:")
print(f"  Modern:       {oxford_modern[0]:.2f}°N, {oxford_modern[1]:.2f}°E")
print(f"  250 Ma Pangaea: {oxford_250ma[0]:.2f}°N, {oxford_250ma[1]:.2f}°E (Subtropical Pangaean interior)\n")

print("Baltimore, USA:")
print(f"  Modern:       {baltimore_modern[0]:.2f}°N, {baltimore_modern[1]:.2f}°E")
print(f"  250 Ma Pangaea: {baltimore_250ma[0]:.2f}°N, {baltimore_250ma[1]:.2f}°E (Central Pangaean suture)")

## 3. Full-Spectrum Optical Filament Mixing for 3D Printing

In multi-material FDM 3D printing (e.g. Bambu AMS, Prusa XL / MMU, or tool-changers), alternating ultra-thin layers (0.08–0.12 mm) of semi-translucent base filaments causes light to pass through adjacent layers, producing a perceived blend of color (optical mixing / pointillism).

### Why Naive RGB Averaging Fails:
Standard slicers that compute colors with naive linear RGB interpolation `(1-t)*C1 + t*C2` fail to model light scattering and absorption:
- Mixing Blue `(0, 33, 133)` and Yellow `(252, 211, 0)` naively gives `(126, 122, 66)` — a **muddy desaturated brown/gray**.
- In physical reality (and under Kubelka-Munk pigment physics), blue absorbs red/yellow light while yellow absorbs blue, allowing green light to reflect — producing **vibrant green**.

`globe3d.filaments` includes a learned degree-4 Kubelka-Munk polynomial model (identical to *Full Spectrum 3D* / *PolyMixer*) that accurately predicts perceived layered blends:

In [ ]:
blue = (0, 33, 133)
yellow = (252, 211, 0)

# Naive linear RGB mix
naive_green = tuple(int(0.5 * b + 0.5 * y) for b, y in zip(blue, yellow))

# Full Spectrum Kubelka-Munk polynomial mix
km_green = mix_filament_pair(blue, yellow, ratio=0.5)

print(f"Blue:           {blue}")
print(f"Yellow:         {yellow}\n")
print(f"Naive RGB mix:  {naive_green} -> Muddy Brown/Gray")
print(f"Full-Spectrum:  {km_green} -> Vibrant Green (47, 141, 56)!")

# Multi-filament blend: Cyan + Yellow + White
c = (0, 255, 255)
y = (255, 255, 0)
w = (250, 250, 250)
meadow_green = mix_filaments([c, y, w], [0.35, 0.50, 0.15])
print(f"Cyan 35% + Yellow 50% + White 15%: {meadow_green} (Meadow Green)")

## 4. Generating a Virtual Filament Palette
From a set of physical spools loaded on your 3D printer (such as a standard 5-spool **CMYKW** bundle: Cyan, Magenta, Yellow, Black, White), `create_filament_palette` generates the entire printable gamut:
- Pure base spools (100%)
- Pairwise mixes across chromatic spools at discrete ratio steps
- Tints (50% mix with White spool)
- Shades (50% mix with Black spool)

In [ ]:
# Generate virtual palette from standard CMYKW spools
palette = create_filament_palette("CMYKW", ratios=[0.25, 0.5, 0.75], include_tints=True, include_shades=True)
print(f"Generated {len(palette)} virtual printable filament colors from 5 physical spools!")

# Display swatch preview of a subset of mixed colors
fig, ax = plt.subplots(figsize=(14, 3.5))
plot_filament_palette(palette[:20], title="Full-Spectrum CMYKW Virtual Filament Palette (Sample Swatches)", ax=ax)
plt.show()

## 5. Setting Up Discrete Banded Colormaps for 3D Printing

Physical 3D printers cannot print continuous analog color gradients without dithering. To print an elevation model, the continuous topography must be partitioned into **discrete elevation bands**, where each band is assigned a specific mixed filament recipe.

`create_elevation_banded_colormap` configures 8 standard geological/hypsometric zones, with physical recipes mixed from base CMYKW spools:

In [ ]:
cmap, norm, band_specs = create_elevation_banded_colormap("CMYKW")

print("Discrete Geological Elevation Bands & Filament Recipes:")
print("-" * 80)
for idx, b in enumerate(band_specs, 1):
    recipe_str = " + ".join(f"{int(w*100)}% {name}" for name, w in b['recipe'])
    print(f"Band {idx}: {b['name']:<24} {b['range']:<18} RGB: {str(b['color']):<16} Recipe: {recipe_str}")

# Preview the discrete banded colorbar
fig, ax = plt.subplots(figsize=(12, 1.8))
thresholds = [-4000, -2000, -200, 0, 500, 1500, 3000]
band_names = [b['name'] for b in band_specs]
plot_banded_colormap_preview(cmap, norm, thresholds, band_labels=band_names, title="Geological Elevation Bands (m)", ax=ax)
plt.show()

## 6. 2D Texture Preprocessing & Rendering

Before wrapping a texture onto a 3D globe, we preprocess it for 3D printing:
1. **Apply Banded Colormap**: Transform the continuous 250 Ma PaleoDEM elevation array into discrete filament color bands using `apply_banded_colormap`.
2. **Undo Hillshading**: Remove synthetic lighting shadows baked into the Scotese PaleoAtlas map texture.
3. **Spherical Vector Stamping**: Stamp reconstructed Oxford and Baltimore star logos, plus North/South pole markers.

In [ ]:
# Load 250 Ma PaleoDEM elevation grid and texture from datasets
dem_grid, texture_path, meta = datasets.paleo.scotese(250.0)
raw_texture = Image.open(texture_path)
print(f"Loaded: {meta['name']} ({meta['age_ma']} Ma)")
print(f"PaleoDEM Grid shape: {dem_grid.data.shape}, range: [{dem_grid.data.min():.0f}m, {dem_grid.data.max():.0f}m]")

# 1. Generate discrete 8-band filament texture
band_colors = [b['color'] for b in band_specs]
banded_texture_arr = apply_banded_colormap(dem_grid.data, band_colors, thresholds)
banded_img = Image.fromarray(banded_texture_arr)

# 2. Preprocess raw texture: remove hillshading
delit_img = undo_hillshading(raw_texture, strength=0.6)

# 3. Stamp reconstructed location markers and pole markers
marked_img = add_svg_symbol_spherical(delit_img, "star", oxford_250ma[0], oxford_250ma[1], symbol_radius_degrees=3.0, fill_color=(255, 0, 0))
marked_img = add_svg_symbol_spherical(marked_img, "star", baltimore_250ma[0], baltimore_250ma[1], symbol_radius_degrees=3.0, fill_color=(255, 255, 0))
marked_img = add_pole_markers(marked_img, marker_size_degrees=6.0, north_symbol="N", south_symbol="S")

# 2D Comparison Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(raw_texture)
axes[0].set_title("1. Raw Scotese PaleoAtlas (250 Ma)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(marked_img)
axes[1].set_title("2. Hillshade-Removed + SVG Star & Pole Glyphs", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(banded_img)
axes[2].set_title("3. Banded Filament Texture (8 Physical CMYKW Mixes)", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 7. Creating the 3D Displaced Globe Model

We construct the 3D spherical mesh with `GlobeModel.from_paleo`:
- `radius=100.0`: 100 mm radius (200 mm diameter globe).
- `n_points=15000`: Mesh vertex density for fast, smooth rendering.
- `coastline_step=1000.0`: 1000 m synthetic shelf vertical step to accentuate ancient shorelines.
- `vert_exagg=35.0`: 35x vertical exaggeration so mountains and deep ocean trenches are physically tactile.
- `hollow=True`: Hollows out the sphere interior to conserve filament.

In [ ]:
model = GlobeModel.from_paleo(
    age=250.0,
    radius=100.0,
    n_points=15000,
    hollow=True,
    inner_ratio=0.9,
    vert_exagg=35.0,
    coastline_step=1000.0,
    mark_poles=True,
    tracked_locations=[
        {"lat": 51.7519, "lon": -1.2578, "name": "Oxford", "symbol": "star", "size": 3.0, "color": (255, 0, 0)},
        {"lat": 39.2904, "lon": -76.6122, "name": "Baltimore", "symbol": "star", "size": 3.0, "color": (255, 255, 0)},
    ]
)

print("Created 250 Ma Pangaea GlobeModel!")
print(f"Outer mesh vertices: {len(model.outer.vertices):,}")
print(f"Inner mesh vertices: {len(model.inner.vertices):,}")
print(f"Radial span: [{np.linalg.norm(model.outer.vertices, axis=1).min():.1f} mm, {np.linalg.norm(model.outer.vertices, axis=1).max():.1f} mm]")

## 8. 3D Mesh Rendering & Perspective Previews

We can render 3D orthographic projections of the colored, displaced spherical mesh using `plot_globe_preview`.
Let's inspect the model from two distinct viewing perspectives:
1. **Pangaea Equatorial View** (Azimuth 45°, Elevation 20°): Centered on the mega-continent Pangaea and the Panthalassa ocean.
2. **North Polar View** (Azimuth 0°, Elevation 75°): Centered on the reconstructed North Pole, showing the stamped 'N' orientation marker.

In [ ]:
fig = plt.figure(figsize=(14, 7))

ax1 = fig.add_subplot(121, projection="3d")
plot_globe_preview(model, elev=20.0, azim=45.0, title="250 Ma Pangaea — Equatorial View", sample_size=12000, point_size=6.0, ax=ax1)

ax2 = fig.add_subplot(122, projection="3d")
plot_globe_preview(model, elev=75.0, azim=0.0, title="250 Ma North Pole — Polar Marker View", sample_size=12000, point_size=6.0, ax=ax2)

plt.tight_layout()
plt.show()

# Note: For interactive WebGL 3D manipulation with mouse rotation and zoom:
# model.preview()

## 9. Magnet Alignment Voids & 3D Print Export

For printing on a standard desktop 3D printer:
1. `configure_magnets()` inserts recessed cylindrical pockets along the equator with alignment bosses.
2. `export_hemispheres()` splits the model along the equator into Northern and Southern hemispheres (with flat equatorial bases for bed adhesion).

In [ ]:
# Configure magnet cavities (e.g. 4 neodymium magnets of 6x3mm along the equator)
# model.configure_magnets(magnet_radius=3.1, magnet_thickness=3.2, count=4)

# Export hemispheres ready for slicer import
# model.export_hemispheres(
#     upper_path="paleo_250ma_north.obj",
#     lower_path="paleo_250ma_south.obj",
#     flip_lower=True,
# )

print("Paleo-globe pipeline complete and ready for physical printing!")